# Health financing and population health in Africa, 2010–2023

This exploratory capstone examines UHC service coverage, life expectancy and under-five mortality alongside health financing. It uses the supplied World Bank export, with 53 countries in the descriptive panel and 52 in the common regression sample. The export omits Djibouti; the country-selection rationale remains undocumented.

**Execution status:** the organized notebook was run from start to finish in a fresh local kernel on 11 November 2026. All 17 code cells passed. The six model, sensitivity and annual-summary exports match the prior Colab run within 1e-8. The original Colab run is preserved alongside it.

Run cells from top to bottom. Local users install requirements.txt first and open this notebook from the repository root or notebooks folder. In Colab, upload the original workbook when prompted. No live data API is required. Unusual mortality and life-expectancy observations are retained. All 84 checked values match the new WDI extract; this establishes WDI consistency, not independent validation of the underlying estimates.


## WDI source-consistency verification
A subsequent extract covering CAF, Somalia and South Sudan, both health outcomes, and 2010–2023 matches all 84 original values exactly. The original values are retained and estimates are unchanged. This comparison completes WDI consistency checking for these records, while independent validation and explanation of the underlying estimates remain separate limitations. See reports/SOURCE_VERIFICATION.md and the full comparison CSV.


## 1. Load the source workbook
The exact source worksheet is `Data (2)`. The input is unchanged. Record the source hash and software versions with each new run. The original download URL, retrieval date and country-selection rationale still need documentation.


In [ ]:
from pathlib import Path
import os
import hashlib
import importlib.metadata
import json
import platform
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_path = Path(os.environ.get("AFRICA_WDI_PATH", str(ROOT / "data/raw/Raw_WDI_Data_Africa.xlsx")))
if not raw_path.is_file():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Place the workbook in data/raw or set AFRICA_WDI_PATH.")
    uploaded = files.upload()
    matches = [name for name in uploaded if name.lower().endswith(".xlsx")]
    if len(matches) != 1:
        raise ValueError("Upload exactly one XLSX source workbook.")
    raw_path = Path(matches[0])
workbook = pd.ExcelFile(raw_path)
TABLES = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
PROCESSED = ROOT / "data/processed"
for folder in [TABLES, FIGURES, PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)
print("Source file:", raw_path.name)
print("Worksheets:", workbook.sheet_names)


In [ ]:
raw_df = pd.read_excel(workbook, sheet_name="Data (2)")

print("Rows and columns:", raw_df.shape)
display(raw_df.head())

## 2. Clean and reshape
Remove footer rows without country/series codes. Convert the fixed 2010–2023 year columns to numeric values. No imputation, interpolation or automatic outlier deletion is applied. The pivot rejects duplicate keys. Maternal mortality and the extra source indicators remain in the panel but are not used in the models.


In [ ]:
# Keep rows with both a country code and an indicator code.
source_df = raw_df.dropna(
    subset=["Country Code", "Series Code"]
).copy()

duplicate_count = source_df.duplicated(
    subset=["Country Code", "Series Code"]
).sum()

print("Source rows:", len(source_df))
print("Countries:", source_df["Country Code"].nunique())
print("Indicators:", source_df["Series Code"].nunique())
print("Duplicate country–indicator records:", duplicate_count)

In [ ]:
if duplicate_count:
    raise ValueError("Duplicate country–indicator records require explicit review.")


In [ ]:
year_columns = [f"{year} [YR{year}]" for year in range(2010, 2024)]

long_df = source_df.melt(
    id_vars=["Country Name", "Country Code", "Series Name", "Series Code"],
    value_vars=year_columns,
    var_name="Year",
    value_name="Value"
)

long_df["Year"] = long_df["Year"].str[:4].astype(int)
original_values = long_df["Value"].copy()
long_df["Value"] = pd.to_numeric(original_values, errors="coerce")
tokens = original_values.astype("string").str.strip()
unexpected = long_df["Value"].isna() & tokens.notna() & ~tokens.isin(["", "..", "...", "NA", "N/A", "NaN"])
if unexpected.any():
    long_df.loc[unexpected].to_csv(TABLES / "unexpected_numeric_tokens.csv", index=False)
    raise ValueError("Review unexpected nonnumeric source values before continuing.")
if np.isinf(long_df["Value"]).any():
    raise ValueError("Infinite source values require review.")

print("Country–indicator–year rows:", len(long_df))

missing_summary = (
    long_df.groupby("Series Name")["Value"]
    .agg(total="size", available="count")
)
missing_summary["missing"] = (
    missing_summary["total"] - missing_summary["available"]
)

display(missing_summary)

In [ ]:
panel_df = long_df.pivot(
    index=["Country Code", "Year"],
    columns="Series Code",
    values="Value"
).reset_index()

country_names = source_df[
    ["Country Code", "Country Name"]
].drop_duplicates()

panel_df = panel_df.merge(
    country_names,
    on="Country Code",
    how="left",
    validate="many_to_one"
)

panel_df.columns.name = None

print("Country–year rows:", len(panel_df))
print("Countries:", panel_df["Country Code"].nunique())
print("Years:", panel_df["Year"].nunique())
print(
    "Duplicate country–year rows:",
    panel_df.duplicated(["Country Code", "Year"]).sum()
)

In [ ]:
analysis_df = panel_df.rename(columns={
    "SH_UHC_SCI": "uhc_service_coverage",
    "SP.DYN.LE00.IN": "life_expectancy",
    "SH.DYN.MORT": "under5_mortality",
    "NY.GDP.PCAP.CD": "gdp_per_capita",
    "SH.XPD.CHEX.PC.CD": "health_expenditure_per_capita",
    "SH.XPD.GHED.GD.ZS": "government_health_expenditure_gdp",
    "SH.XPD.OOPC.CH.ZS": "out_of_pocket_expenditure"
}).copy()

main_variables = [
    "uhc_service_coverage",
    "life_expectancy",
    "under5_mortality",
    "gdp_per_capita",
    "health_expenditure_per_capita",
    "government_health_expenditure_gdp",
    "out_of_pocket_expenditure"
]

print(analysis_df[main_variables].describe().T.to_string())

## 3. Source observations requiring verification
The workbook records CAF under-five mortality of 424.6 per 1,000 and life expectancy of 18.818 years in 2022. Other unusual paired values occur in CAF, Somalia and South Sudan. These are source questions, not confirmed errors. A high value or IQR flag alone does not justify deletion. The failed external API check has been removed from the execution path; no replacement values have been introduced. The five-record exclusion below reproduces a historical influence check, not an exhaustive anomaly screen.


In [ ]:
caf_mortality = panel_df.loc[
    panel_df["Country Code"].eq("CAF"),
    ["Year", "SH.DYN.MORT"]
].rename(columns={
    "SH.DYN.MORT": "Under-five deaths per 1,000 live births"
})

display(caf_mortality.sort_values("Year"))

In [ ]:
print("Ten lowest life-expectancy observations:")

print(
    analysis_df.nsmallest(10, "life_expectancy")[
        [
            "Country Name",
            "Year",
            "life_expectancy",
            "under5_mortality"
        ]
    ].to_string(index=False)
)

In [ ]:
metadata_df = pd.read_excel(
    workbook,
    sheet_name="Series - Metadata"
)

selected_metadata = metadata_df.loc[
    metadata_df["Code"].isin(["SP.DYN.LE00.IN", "SH.DYN.MORT"]),
    ["Code", "Indicator Name", "Source", "Reference period",
     "Limitations and exceptions", "General comments"]
]

print(selected_metadata.to_string(index=False))

## 4. Common regression sample
Use positive-value logs for GDP and spending. The primary models share complete cases across all seven analytical variables, to keep model comparisons on the same rows. Expected source-specific result: 713 observations, 52 countries, 14 years. Exclusions: Eritrea 12 rows, Somalia 3, South Sudan 14. Annual sample counts are 51 except 2012 (50). Selection may affect generalizability.


In [ ]:
import numpy as np

# Log only positive values; leave missing/nonpositive values missing.
for variable in ["gdp_per_capita", "health_expenditure_per_capita"]:
    analysis_df[f"log_{variable}"] = np.log(
        analysis_df[variable].where(analysis_df[variable] > 0)
    )

regression_variables = [
    "uhc_service_coverage",
    "life_expectancy",
    "under5_mortality",
    "log_gdp_per_capita",
    "log_health_expenditure_per_capita",
    "government_health_expenditure_gdp",
    "out_of_pocket_expenditure"
]

regression_df = analysis_df.dropna(
    subset=regression_variables
).copy()

print("Regression observations:", len(regression_df))
print("Countries:", regression_df["Country Code"].nunique())
print("Years:", regression_df["Year"].nunique())

print("\nExcluded country–year rows:")
excluded = analysis_df.loc[
    ~analysis_df.index.isin(regression_df.index)
]
print(excluded.groupby("Country Name").size().to_string())

## 5. Country and year fixed-effects models
Each model includes log GDP per capita and one financing variable, with country and year indicator terms. Standard errors are clustered by country, with finite-sample correction and t inference using 51 cluster degrees of freedom. All nine financing comparisons are exploratory and p-values are unadjusted.

Country effects control for time-invariant country differences; year effects control for shocks common to all countries. Time-varying confounding, reverse causation and measurement error remain possible. Country clustering does not resolve arbitrary dependence across countries. These are associations, not causal effects.

For level outcomes with logged spending, a 10% spending increase corresponds to coefficient × log(1.10) outcome units. Government and out-of-pocket coefficients refer to one percentage-point changes in their respective shares, which have different denominators.


In [ ]:
outcomes = [
    "uhc_service_coverage",
    "life_expectancy",
    "under5_mortality"
]

financing_variables = [
    "log_health_expenditure_per_capita",
    "government_health_expenditure_gdp",
    "out_of_pocket_expenditure"
]

fitted_models = {}
results = []

for outcome in outcomes:
    for financing in financing_variables:
        model = smf.ols(
            f"{outcome} ~ log_gdp_per_capita + {financing} "
            "+ C(Q('Country Code')) + C(Year)",
            data=regression_df
        ).fit(
            cov_type="cluster",
            cov_kwds={"groups": regression_df["Country Code"], "use_correction": True, "df_correction": True},
            use_t=True
        )

        fitted_models[(outcome, financing)] = model
        ci = model.conf_int().loc[financing]

        results.append({
            "Outcome": outcome,
            "Financing indicator": financing,
            "Coefficient": model.params[financing],
            "P-value": model.pvalues[financing],
            "95% CI lower": ci.iloc[0],
            "95% CI upper": ci.iloc[1],
            "Observations": int(model.nobs)
        })

model_results = pd.DataFrame(results)
print(model_results.round(4).to_string(index=False))

The received Colab run reproduces all nine historical financing coefficients. Every financing 95% interval includes zero. This is insufficient evidence of an association under these specifications, not proof of no relationship or evidence that financing is unimportant. All 84 checked source values match the new WDI extract. Interpretation still requires caution because the underlying estimates are unusual and the mortality coefficient is sensitive to influential records. No claim of out-of-sample predictive performance is made.


## 6. Historical mortality influence check
Retain all valid data for the primary analysis. Repeat the original exclusion of CAF 2014/2019/2021/2022 and SLE 2010 only to assess influence. SLE 2010 is not established to be a source error; this list was inherited from the earlier analysis and does not cover all unusual life-expectancy or mortality records.


In [ ]:
flagged_records = [
    ("CAF", 2014),
    ("CAF", 2019),
    ("CAF", 2021),
    ("CAF", 2022),
    ("SLE", 2010)
]

flagged = regression_df.set_index(
    ["Country Code", "Year"]
).index.isin(flagged_records)

sensitivity_df = regression_df.loc[~flagged].copy()

sensitivity_model = smf.ols(
    "under5_mortality ~ log_gdp_per_capita "
    "+ log_health_expenditure_per_capita "
    "+ C(Q('Country Code')) + C(Year)",
    data=sensitivity_df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": sensitivity_df["Country Code"]},
    use_t=True
)

comparison_rows = []

for label, model in [
    (
        "Full sample",
        fitted_models[
            ("under5_mortality", "log_health_expenditure_per_capita")
        ]
    ),
    ("Historical five-record exclusion", sensitivity_model)
]:
    term = "log_health_expenditure_per_capita"
    ci = model.conf_int().loc[term]

    comparison_rows.append({
        "Sample": label,
        "Observations": int(model.nobs),
        "Spending coefficient": model.params[term],
        "P-value": model.pvalues[term],
        "95% CI lower": ci.iloc[0],
        "95% CI upper": ci.iloc[1]
    })

sensitivity_results = pd.DataFrame(comparison_rows)
print(sensitivity_results.round(4).to_string(index=False))

The spending coefficient changes from +7.3335 to −0.4913 after five exclusions (713 to 708 rows). Both intervals include zero, but the sign and magnitude change substantially. Do not describe the coefficient as robust merely because neither p-value is below .05.


## 7. Log-mortality sensitivity
Retain positive mortality values and model their natural logarithm. This changes the functional form, not the source observations. The spending coefficient is an elasticity in this specification.


In [ ]:
log_mortality_df = regression_df.loc[
    regression_df["under5_mortality"] > 0
].copy()

log_mortality_df["log_under5_mortality"] = np.log(
    log_mortality_df["under5_mortality"]
)

log_mortality_model = smf.ols(
    "log_under5_mortality ~ log_gdp_per_capita "
    "+ log_health_expenditure_per_capita "
    "+ C(Q('Country Code')) + C(Year)",
    data=log_mortality_df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": log_mortality_df["Country Code"]},
    use_t=True
)

term = "log_health_expenditure_per_capita"
ci = log_mortality_model.conf_int().loc[term]

log_mortality_results = pd.DataFrame([{
    "Observations": int(log_mortality_model.nobs),
    "Spending coefficient": log_mortality_model.params[term],
    "P-value": log_mortality_model.pvalues[term],
    "95% CI lower": ci.iloc[0],
    "95% CI upper": ci.iloc[1]
}])

print(log_mortality_results.round(4).to_string(index=False))

The received run gives an elasticity of 0.0305 (95% CI −0.0918 to 0.1528; p=.6189). The interval includes zero. A log transformation does not validate the unusual source observations.


## 8. Descriptive figures
The same common regression sample is used in both figures. These are unweighted country means with changing membership, not population-weighted estimates for Africa.


In [ ]:
import matplotlib.pyplot as plt

outcome_labels = {
    "uhc_service_coverage": "UHC service coverage index",
    "life_expectancy": "Life expectancy (years)",
    "under5_mortality": "Under-five deaths per 1,000 live births"
}

annual_means = regression_df.groupby("Year")[
    list(outcome_labels)
].mean()

annual_counts = regression_df.groupby("Year").size()

fig, axes = plt.subplots(3, 1, figsize=(10, 10))

for ax, (variable, label) in zip(axes, outcome_labels.items()):
    ax.plot(annual_means.index, annual_means[variable], marker="o")
    ax.set_ylabel(label)
    ax.set_xlabel("Year")
    ax.grid(alpha=0.25)

fig.suptitle(
    "Health outcomes, 2010–2023\n"
    "Unweighted country means in the regression sample",
    fontsize=13
)

fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIGURES / "health_outcome_trends.png", dpi=200, bbox_inches="tight")
plt.show()

print("Countries included each year:")
print(annual_counts.to_string())

These figures show unweighted annual country means for the complete-case regression sample: 51 countries per year, except 2012 (50). Country membership varies across years. Overall, UHC coverage and life expectancy increased while under-five mortality declined. Unusual source observations match the new WDI extract; their demographic interpretation is not independently validated, so individual annual fluctuations should be interpreted cautiously. These descriptive trends do not establish that health financing caused the changes.

In [ ]:
financing_labels = {
    "health_expenditure_per_capita":
        "Health spending per capita (current US$)",
    "government_health_expenditure_gdp":
        "Government health spending (% of GDP)",
    "out_of_pocket_expenditure":
        "Out-of-pocket spending (% of current health expenditure)"
}

annual_financing = regression_df.groupby("Year")[
    list(financing_labels)
].mean()

fig, axes = plt.subplots(3, 1, figsize=(11, 10))

for ax, (variable, label) in zip(axes, financing_labels.items()):
    ax.plot(
        annual_financing.index,
        annual_financing[variable],
        marker="o"
    )
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Year")
    ax.set_ylabel("Current US$" if variable == "health_expenditure_per_capita"
                  else "Percent")
    ax.grid(alpha=0.25)

fig.suptitle(
    "Health financing, 2010–2023\n"
    "Unweighted country means in the regression sample",
    fontsize=13
)

fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIGURES / "health_financing_trends.png", dpi=200, bbox_inches="tight")
plt.show()

These figures show unweighted annual country means for the regression sample: 51 countries per year, except 2012 (50). Health spending per capita fluctuated but ended above its 2010 level. Government health spending as a share of GDP peaked in 2020, while the out-of-pocket share ended below its 2010 level.

Per-capita spending is measured in current US dollars and is not adjusted for inflation or exchange-rate changes. A falling out-of-pocket share does not necessarily mean households paid less in absolute terms or experienced less financial hardship. Country membership varies across years, and these descriptive trends do not establish causal relationships.

## 9. Export results and provenance
Write files once at the end. The input hash identifies the source file; the run manifest records package versions without claiming the environment is a validated dependency lockfile.


In [ ]:
tables = {
    "main_regression_results.csv": model_results,
    "mortality_exclusion_sensitivity.csv": sensitivity_results,
    "log_mortality_sensitivity.csv": log_mortality_results,
    "annual_health_outcome_means.csv": annual_means,
    "annual_financing_means.csv": annual_financing,
    "annual_country_counts.csv": annual_counts.rename("countries").to_frame()
}
for name, table in tables.items():
    table.to_csv(TABLES / name, index=name.startswith("annual_"))
regression_df.to_csv(PROCESSED / "regression_sample.csv", index=False)
analysis_df.to_csv(PROCESSED / "analysis_panel.csv", index=False)
excluded.to_csv(TABLES / "excluded_country_years.csv", index=False)
missing_summary.to_csv(TABLES / "missingness_by_indicator.csv")
country_names.to_csv(TABLES / "country_roster.csv", index=False)
metadata_df.to_csv(TABLES / "source_metadata.csv", index=False)
manifest = {
    "source_sha256": hashlib.sha256(raw_path.read_bytes()).hexdigest(),
    "worksheet": "Data (2)",
    "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ["numpy", "pandas", "matplotlib", "statsmodels", "openpyxl"]}
}
(ROOT / "results/run_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Results saved in the project results folder.")


## Conclusion and remaining work
The model estimates and sensitivities reproduce the received Colab results. Financing coefficients are imprecise under these specifications, and mortality estimates are sensitive to influential records. The requested WDI comparison is complete: all 84 checked values match. Further explanation of the underlying estimates, country selection and original acquisition provenance remain limitations. Do not silently replace the workbook with a newer data vintage. A later source correction requires rerunning affected models and comparing versions.
